# Diabetes Prediction Using Machine Learning



## Preprocessing Workflow

```mermaid
flowchart TD
    A[Load raw CSV] --> B[Verify expected columns]
    B --> C[Create working copy]
    C --> D[Check duplicate rows]
    D --> E[Replace impossible zeros with NaN in selected columns]
    E --> F[Count missing values before imputation]
    F --> G[Median imputation]
    G --> H[Verify no missing values remain]
    H --> I[Separate X and y]
    I --> J[Scale input features only]
    J --> K[Save cleaned and scaled datasets]
    K --> L[Save fitted imputer and scaler]
```


## 1. Import Required Libraries



In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


## 2. Load Dataset Using pathlib



In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_RELATIVE_PATH = Path("data") / "raw" / "diabetes.csv"
PROCESSED_RELATIVE_DIR = Path("data") / "processed"
MODELS_RELATIVE_DIR = Path("models")
WEEK6_TABLES_RELATIVE_DIR = Path("outputs") / "tables" / "week_06"

DATASET_PATH = PROJECT_ROOT / DATASET_RELATIVE_PATH
PROCESSED_DIR = PROJECT_ROOT / PROCESSED_RELATIVE_DIR
MODELS_DIR = PROJECT_ROOT / MODELS_RELATIVE_DIR
WEEK6_TABLES_DIR = PROJECT_ROOT / WEEK6_TABLES_RELATIVE_DIR

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset file not found: {DATASET_RELATIVE_PATH}")

df = pd.read_csv(DATASET_PATH)
print("Dataset loaded successfully.")
print(f"Dataset path: {DATASET_RELATIVE_PATH}")
display(df.head())


Dataset loaded successfully.
Dataset path: data\raw\diabetes.csv


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 3. Verify Expected Columns




In [3]:
EXPECTED_COLUMNS = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome",
]

missing_columns = [column for column in EXPECTED_COLUMNS if column not in df.columns]
extra_columns = [column for column in df.columns if column not in EXPECTED_COLUMNS]

if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

if extra_columns:
    print(f"Extra columns found: {extra_columns}")
else:
    print("All expected columns are present.")


All expected columns are present.


## 4. Create a Working Copy

A working copy is created before preprocessing. This protects the original file `data/raw/diabetes.csv` from accidental modification.


In [4]:
working_df = df.copy()
print("Working copy created.")
print(f"Original dataset shape: {df.shape}")
print(f"Working copy shape: {working_df.shape}")


Working copy created.
Original dataset shape: (768, 9)
Working copy shape: (768, 9)


## 5. Check Duplicate Rows




In [6]:
duplicate_count = int(working_df.duplicated().sum())

duplicate_summary = pd.DataFrame({
    "check": ["duplicate_rows"],
    "count": [duplicate_count],
    "action": [
        "Checked only; no duplicate rows removed because count is zero"
        if duplicate_count == 0
        else "Documented only; review before removal"
    ],
})

display(duplicate_summary)


,check,count,action
0,duplicate_rows,0,Checked only; no duplicate rows removed becaus...


## 6. Replace Medically Impossible Zero Values with NaN

Zero is medically impossible or suspicious for `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI`. These zeros are replaced with `NaN` so they can be imputed properly.

Zeros are not replaced in `Pregnancies`, `Outcome`, `Age`, or `DiabetesPedigreeFunction` because zero can be valid or meaningful for those columns, or those columns do not contain zero values needing this treatment.


In [7]:
ZERO_AS_MISSING_COLUMNS = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
DO_NOT_REPLACE_ZERO_COLUMNS = ["Pregnancies", "Outcome", "Age", "DiabetesPedigreeFunction"]

explicit_missing_before = working_df.isna().sum()
zero_counts_before = pd.Series({column: int((working_df[column] == 0).sum()) for column in EXPECTED_COLUMNS})

working_df[ZERO_AS_MISSING_COLUMNS] = working_df[ZERO_AS_MISSING_COLUMNS].replace(0, np.nan)

missing_before = pd.DataFrame({
    "column_name": working_df.columns,
    "explicit_missing_before_zero_replacement": [int(explicit_missing_before[column]) for column in working_df.columns],
    "zero_count_before_replacement": [int(zero_counts_before[column]) for column in working_df.columns],
    "missing_count_before_imputation": working_df.isna().sum().values,
})

display(missing_before)


,column_name,explicit_missing_before_zero_replacement,zero_count_before_replacement,missing_count_before_imputation
0,Pregnancies,0,111,0
1,Glucose,0,5,5
2,BloodPressure,0,35,35
3,SkinThickness,0,227,227
4,Insulin,0,374,374
5,BMI,0,11,11
6,DiabetesPedigreeFunction,0,0,0
7,Age,0,0,0
8,Outcome,0,500,0


## 7. Median Imputation

Median imputation fills missing values using the middle value of each column. Median is selected because health measurements such as insulin and BMI can contain extreme values. The median is less affected by extreme values than the mean, so it is a safer simple choice for this project. A fitted `SimpleImputer` is also created with the same median strategy so the preprocessing step can be reused later.


In [8]:
median_values = working_df[ZERO_AS_MISSING_COLUMNS].median()
median_imputer = SimpleImputer(strategy="median")
median_imputer.fit(working_df[ZERO_AS_MISSING_COLUMNS])

cleaned_df = working_df.copy()
cleaned_df[ZERO_AS_MISSING_COLUMNS] = cleaned_df[ZERO_AS_MISSING_COLUMNS].fillna(median_values)

median_imputation_values = median_values.reset_index()
median_imputation_values.columns = ["column_name", "median_used_for_imputation"]

display(median_imputation_values)


,column_name,median_used_for_imputation
0,Glucose,117.0
1,BloodPressure,72.0
2,SkinThickness,29.0
3,Insulin,125.0
4,BMI,32.3


## 8. Verify No Missing Values Remain

After median imputation, all columns are checked again to confirm that no missing values remain.


In [9]:
missing_after = pd.DataFrame({
    "column_name": cleaned_df.columns,
    "missing_count_after_imputation": cleaned_df.isna().sum().values,
})

display(missing_after)

remaining_missing_values = int(cleaned_df.isna().sum().sum())
if remaining_missing_values != 0:
    raise ValueError(f"Missing values still remain after imputation: {remaining_missing_values}")

print("No missing values remain after median imputation.")


,column_name,missing_count_after_imputation
0,Pregnancies,0
1,Glucose,0
2,BloodPressure,0
3,SkinThickness,0
4,Insulin,0
5,BMI,0
6,DiabetesPedigreeFunction,0
7,Age,0
8,Outcome,0


No missing values remain after median imputation.


## 9. Separate Input Features and Target Variable

The input features are stored in `X`, and the target variable is stored in `y`. The target variable is not scaled.


In [10]:
target_variable = "Outcome"
X = cleaned_df.drop(columns=target_variable)
y = cleaned_df[target_variable]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print("Input features:", list(X.columns))
print("Target variable:", target_variable)


X shape: (768, 8)
y shape: (768,)
Input features: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
Target variable: Outcome


## 10. Apply StandardScaler on Input Features Only

`StandardScaler` converts input features to a standard scale with mean around 0 and standard deviation around 1. It is applied only to `X`. The target variable `Outcome` remains unchanged because it is a class label, not an input measurement.


In [11]:
scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(X)

scaled_df = pd.DataFrame(X_scaled_array, columns=X.columns, index=X.index)
scaled_df[target_variable] = y.values

print("Scaled dataset preview:")
display(scaled_df.head())

print("Target values after scaling step:")
print(scaled_df[target_variable].value_counts().sort_index())


Scaled dataset preview:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,0.639947,0.866045,-0.031990,0.670643,-0.181541,0.166619,0.468492,1.425995,1
1,-0.844885,-1.205066,-0.528319,-0.012301,-0.181541,-0.852200,-0.365061,-0.190672,0
2,1.233880,2.016662,-0.693761,-0.012301,-0.181541,-1.332500,0.604397,-0.105584,1
3,-0.844885,-1.073567,-0.528319,-0.695245,-0.540642,-0.633881,-0.920763,-1.041549,0
4,-1.141852,0.504422,-2.679076,0.670643,0.316566,1.549303,5.484909,-0.020496,1


Target values after scaling step:
Outcome
0    500
1    268
Name: count, dtype: int64


## 11. Save Cleaned and Scaled Datasets

The cleaned dataset contains median-imputed values but unscaled input features. The scaled dataset contains standardized input features and the original unscaled target variable.


In [12]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

cleaned_path = PROCESSED_DIR / "diabetes_cleaned.csv"
scaled_path = PROCESSED_DIR / "diabetes_scaled.csv"

cleaned_df.to_csv(cleaned_path, index=False)
scaled_df.to_csv(scaled_path, index=False)

print(f"Cleaned dataset saved to: {PROCESSED_RELATIVE_DIR / cleaned_path.name}")
print(f"Scaled dataset saved to: {PROCESSED_RELATIVE_DIR / scaled_path.name}")


Cleaned dataset saved to: data\processed\diabetes_cleaned.csv
Scaled dataset saved to: data\processed\diabetes_scaled.csv


## 12. Save Fitted Preprocessing Artifacts

The fitted `SimpleImputer` and `StandardScaler` are saved with joblib. These artifacts can be reused later so future data is preprocessed in the same way.


In [13]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

scaler_path = MODELS_DIR / "scaler.pkl"
imputer_path = MODELS_DIR / "median_imputer.pkl"

joblib.dump(scaler, scaler_path)
joblib.dump(median_imputer, imputer_path)

if not scaler_path.exists():
    raise FileNotFoundError(f"Scaler artifact was not created: {scaler_path}")

if not imputer_path.exists():
    raise FileNotFoundError(f"Median imputer artifact was not created: {imputer_path}")

print(f"Scaler artifact saved to: {MODELS_RELATIVE_DIR / scaler_path.name}")
print(f"Median imputer artifact saved to: {MODELS_RELATIVE_DIR / imputer_path.name}")


Scaler artifact saved to: models\scaler.pkl
Median imputer artifact saved to: models\median_imputer.pkl


## 13. Save Summary Tables





In [19]:
from pathlib import Path
import pandas as pd

# --------------------------------------------------
# 1. Define project paths
# --------------------------------------------------

PROJECT_ROOT = Path.cwd()

# If running the notebook from the "notebooks" folder,
# move one level up to the project root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_TABLES_RELATIVE_DIR = (
    Path("outputs") / "tables" 
)

OUTPUT_TABLES_DIR = (
    PROJECT_ROOT / OUTPUT_TABLES_RELATIVE_DIR
)

# Create output directory if it does not exist
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------
# 2. Create preprocessing summary
# --------------------------------------------------

preprocessing_summary = pd.DataFrame([
    {
        "step": "Create working copy",
        "columns_affected": "All columns",
        "decision": "Used df.copy() so the raw dataset remains unchanged",
        "output": "working_df",
    },
    {
        "step": "Duplicate check",
        "columns_affected": "All columns",
        "decision": f"Duplicate rows found: {duplicate_count}",
        "output": "duplicate_summary.csv",
    },
    {
        "step": "Impossible zero replacement",
        "columns_affected": ", ".join(ZERO_AS_MISSING_COLUMNS),
        "decision": (
            "Replaced zeros with NaN only in medically "
            "impossible measurement columns"
        ),
        "output": "working_df with NaN values",
    },
    {
        "step": "Median imputation",
        "columns_affected": ", ".join(ZERO_AS_MISSING_COLUMNS),
        "decision": (
            "Used median because it is less affected "
            "by extreme values than the mean"
        ),
        "output": "data/processed/diabetes_cleaned.csv",
    },
    {
        "step": "Separate X and y",
        "columns_affected": "Input columns and Outcome",
        "decision": (
            "Separated input features from the target "
            "variable before scaling"
        ),
        "output": "X and y",
    },
    {
        "step": "Standard scaling",
        "columns_affected": ", ".join(X.columns),
        "decision": (
            "Applied StandardScaler only to input features; "
            "Outcome was not scaled"
        ),
        "output": "data/processed/diabetes_scaled.csv",
    },
    {
        "step": "Save fitted preprocessing artifacts",
        "columns_affected": "Input features",
        "decision": (
            "Saved fitted SimpleImputer and StandardScaler "
            "using joblib for later reuse"
        ),
        "output": (
            "models/median_imputer.pkl and models/scaler.pkl"
        ),
    },
])


# --------------------------------------------------
# 3. Tables to save
# --------------------------------------------------

tables_to_save = {
    "duplicate_summary.csv": duplicate_summary,
    "missing_before.csv": missing_before,
    "missing_after.csv": missing_after,
    "preprocessing_summary.csv": preprocessing_summary,
    "median_imputation_values.csv": median_imputation_values,
}


# --------------------------------------------------
# 4. Save tables
# --------------------------------------------------

for file_name, table in tables_to_save.items():

    output_path = OUTPUT_TABLES_DIR / file_name

    table.to_csv(output_path, index=False)


# --------------------------------------------------
# 5. Display saved files
# --------------------------------------------------

print("Summary tables saved successfully:\n")

for file_name in tables_to_save:
    print(f"- {OUTPUT_TABLES_RELATIVE_DIR / file_name}")


Summary tables saved successfully:

- outputs\tables\duplicate_summary.csv
- outputs\tables\missing_before.csv
- outputs\tables\missing_after.csv
- outputs\tables\preprocessing_summary.csv
- outputs\tables\median_imputation_values.csv
